# California Housing — End-to-End Regression Workflow

## The standard workflow

**1. PROBLEM → 2. UNDERSTAND → 3. SPLIT → 4. PREPARE → 5. FIT → 6. PREDICT → 7. EVALUATE**

Use one question throughout: **Which job am I doing now, and why?**

Across the workflow: **Question → Evidence → Decision**.

## 1. Load the data

Start with the data and the prediction question, not with an algorithm.

One row represents a California district / census block group.

Our target is:

`median_house_value`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

housing = pd.read_csv("california_housing.csv")

housing.head()

## 2. Inspect before changing anything

Preprocessing should solve problems we actually observe.

Use `info()` to check:

- number of rows and columns
- data types
- missing values

In [ ]:
housing.info()

### Interpretation

- `total_bedrooms` has missing values.
- `ocean_proximity` is categorical text.
- the other predictors are numerical.

These observations will later justify imputation and one-hot encoding.

## 3. Understand numerical ranges

`describe()` gives exact numerical landmarks.

For `median_income`, remember the unit:

- 3.0 ≈ $30,000
- 5.0 ≈ $50,000

In [ ]:
housing.describe().round(2)

### Interpretation

At this stage, the maximum recorded house value is `$500,001`.

Do not immediately call it a cap just from `describe()`.
Say:

> “This is an unusual upper value worth investigating in the distribution.”

## 4. Inspect the categorical feature

In [ ]:
housing["ocean_proximity"].value_counts()

### Key idea

These labels are categories, not ordered quantities.

So we should not replace them with `1, 2, 3, 4, 5`, because that would create an artificial numerical order.

# STEP 2 — UNDERSTAND: EDA driven by questions

## 5. What do the numerical distributions look like?

Are values concentrated, skewed, or piled up at unusual limits?

In [ ]:
housing.hist(
    bins=20,
    figsize=(14, 10)
)

plt.tight_layout()
plt.show()

### Interpretation

- several variables are right-skewed
- the target shows many values near the upper limit
- the histogram gives us more information about the unusual `$500,001` maximum

## 6. Which numerical features are most related to house value?

Correlation is a clue about linear association, not proof of causation.

In [ ]:
correlations = (
    housing.corr(numeric_only=True)["median_house_value"]
    .sort_values(ascending=False)
    .round(2)
)

correlations

### Interpretation

`median_income` has the strongest positive numerical correlation with house value.

This gives us a reason to examine income more closely.

## 7. Does house value increase with income?

In [ ]:
plt.figure(figsize=(9, 6))

plt.scatter(
    housing["median_income"],
    housing["median_house_value"],
    alpha=0.8
)

plt.xlabel("Median income ($10,000 units)")
plt.ylabel("Median house value ($)")
plt.title("Median income and house value")
plt.show()

### Key idea

The cloud generally rises, so higher-income districts tend to have higher house values.

But the relationship is not perfect. Income is useful, not sufficient.

## 8. Does house value vary geographically?

Here:

- x + y = **where**
- color = **house value**

In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    housing["longitude"],
    housing["latitude"],
    c=housing["median_house_value"],
    cmap="turbo",
    s=15,
    alpha=1
)

plt.xlabel("Longitude", fontsize=16)
plt.ylabel("Latitude", fontsize=16)
plt.title("California districts: location and median house value", fontsize=16)
cbar = plt.colorbar()
cbar.set_label("Median house value ($)", fontsize=16)
plt.show()

## 9. Combine location, population and house value

This is the more information-rich plot shown after the simple geographic plot.

- position = location
- marker size = population
- color = house value

In [ ]:
plt.figure(figsize=(9, 7))

plt.scatter(
    housing["longitude"],
    housing["latitude"],
    s=housing["population"] / 50,
    c=housing["median_house_value"],
    cmap="turbo",
    alpha=1
)

plt.xlabel("Longitude", fontsize=16)
plt.ylabel("Latitude", fontsize=16)
plt.title("Location, population and median house value", fontsize=16)
cbar = plt.colorbar()
cbar.set_label("Median house value ($)", fontsize=16)
plt.show()

## 10. EDA → decisions

At this point, connect observations to modeling decisions:

| Observation | Decision |
|---|---|
| missing `total_bedrooms` | impute missing numeric values |
| text `ocean_proximity` | one-hot encode categories |
| very different numeric scales | use a numeric scaling step |
| income strongly related to target | preserve broad income composition in the split |
| several variables contain useful signal | use multiple predictors in the model |

This is the bridge from **exploration** to **workflow design**.

# STEP 3 — SPLIT: Protect the test set

## 11. Why split now?

The test set is the final evaluation set.

- **training set** → learn preprocessing rules and model parameters
- **test set** → evaluate on unseen data

We split before learning preprocessing values.

## 12. Understand income before choosing strata

First look at numerical landmarks.

In [ ]:
income = housing["median_income"]

income_landmarks = pd.Series({
    "Minimum": income.min(),
    "Q1": income.quantile(0.25),
    "Median": income.median(),
    "Q3": income.quantile(0.75),
    "Maximum": income.max()
})

income_landmarks

Now look at the shape.

Quartiles, histogram bins and stratification groups are different ideas.

- quartiles summarize the distribution
- histogram bins visualize the distribution
- stratification groups are a practical sampling design choice

In [ ]:
plt.figure(figsize=(9, 4))

plt.hist(income, bins=30)

plt.xlabel("Median income ($10,000 units)")
plt.ylabel("Number of districts")
plt.title("Distribution of median income")

plt.show()


# Quartiles
income = housing["median_income"].dropna()

# Quartiles
q1 = income.quantile(0.25)
median = income.quantile(0.50)
q3 = income.quantile(0.75)

# Interquartile range
iqr = q3 - q1

# Outlier boundaries: 1.5 × IQR rule
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Find observations outside the boundaries
outliers = income[
    (income < lower_bound) |
    (income > upper_bound)
]

print("Minimum:", income.min())
print("Q1 (25%):", q1)
print("Median (50%):", median)
print("Q3 (75%):", q3)
print("Maximum:", income.max())

print("\nIQR:", iqr)
print("Lower boundary:", lower_bound)
print("Upper boundary:", upper_bound)

print("\nNumber of outliers:", len(outliers))
print("Outlier values:")
print(outliers.values)


# Box plot
plt.figure(figsize=(9, 4))
plt.boxplot(income, vert=False)

plt.xlabel("Median income ($10,000 units)")
plt.title("Median income: quartiles and outliers")

plt.show()

## 13. Create temporary income groups

We use broad groups:

- 0–1.5
- 1.5–3.0
- 3.0–4.5
- 4.5–6.0
- 6.0+

These are not mathematically unique or “the correct bins.”

They are a practical compromise: meaningful broad income ranges with a merged sparse upper tail.

In [ ]:
housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5]
)

income_summary = pd.DataFrame({
    "Count": housing["income_cat"].value_counts().sort_index(),
    "Percent": (
        housing["income_cat"]
        .value_counts(normalize=True)
        .sort_index() * 100
    ).round(1)
})

income_summary

In [ ]:
income_summary["Count"].plot(
    kind="bar",
    figsize=(8, 5)
)

plt.xlabel("Temporary income category")
plt.ylabel("Number of districts")
plt.title("Districts in each temporary income group")
plt.xticks(rotation=0)
plt.show()

### Key idea

The groups do **not** need equal numbers of districts.

Stratification means:

> preserve the proportion of each chosen group in train and test.

## 14. Make the stratified 80/20 split

In [ ]:
from sklearn.model_selection import train_test_split

train_set, test_set = train_test_split(
    housing,
    test_size=0.20,
    random_state=42,
    stratify=housing["income_cat"]
)

print("Training rows:", len(train_set))
print("Test rows:", len(test_set))

## 15. Verify that the proportions were preserved

In [ ]:
comparison = pd.DataFrame({
    "Overall (%)": (
        housing["income_cat"]
        .value_counts(normalize=True)
        .sort_index() * 100
    ),
    "Train (%)": (
        train_set["income_cat"]
        .value_counts(normalize=True)
        .sort_index() * 100
    ),
    "Test (%)": (
        test_set["income_cat"]
        .value_counts(normalize=True)
        .sort_index() * 100
    )
})

comparison.round(1)

`income_cat` was only a helper for splitting.

Keep the original `median_income`, but remove the artificial helper column before modeling.

In [ ]:
train_set = train_set.drop(columns=["income_cat"])
test_set = test_set.drop(columns=["income_cat"])

# STEP 4 — PREPARE: Make the features usable

## 16. Separate features from target

- `X` = what the model receives
- `y` = what the model should predict

In [ ]:
X_train = train_set.drop(columns=["median_house_value"])
y_train = train_set["median_house_value"].copy()

X_test = test_set.drop(columns=["median_house_value"])
y_test = test_set["median_house_value"].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

## 17. Identify numerical and categorical columns

In [ ]:
numeric_cols = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

## DEFINE → LEARN → APPLY

Before using the Pipeline, separate these three ideas:

- **DEFINE**: `Pipeline(...)`, `ColumnTransformer(...)`, `SimpleImputer(...)` describe the recipe. No values have been learned yet.
- **LEARN**: `model.fit(X_train, y_train)` learns preprocessing values **and** model parameters from the training data.
- **APPLY**: `model.predict(X_test)` applies the training-learned preparation to `X_test` and then predicts.

> **TRAIN teaches. TEST checks.**

## 18. Before Pipeline: `fit()` and `transform()`

> **fit = learn**
> **transform = apply**

Examples:

- Imputer `fit` → learn the training median
- Scaler `fit` → learn training mean and spread
- Encoder `fit` → learn training categories

Then `transform` applies those learned rules.

### Rule

**TRAIN → learn + apply**
**TEST → apply only**

### Concrete example

If training bedrooms are:

`100, 120, missing, 140`

the imputer learns median = `120`.

A missing bedroom value in the test set should also be filled with **120**.

We do not calculate a new median from the test set.

## 19. Why Pipeline?

We could call each preprocessing function separately.

Pipeline simply **remembers the sequence**.

For numerical data:

`Imputer → Scaler`

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_pipeline

### Interpretation

> “Whenever numerical data arrive, first fill missing values, then scale the result.”

Individual functions do the work.
The Pipeline keeps the recipe together.

## 20. Categorical path

`ocean_proximity` needs a different transformation:

`OneHotEncoder`

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

categorical_pipeline

## 21. Why ColumnTransformer?

Pipeline answers:

> **In what order should steps happen?**

ColumnTransformer answers:

> **Which columns should receive which steps?**

So:

- numerical columns → Imputer → Scaler
- categorical columns → OneHotEncoder

In [ ]:
from sklearn.compose import ColumnTransformer

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

preprocess

### Read the code as English

`("num", numeric_pipeline, numeric_cols)`

means:

> “Send the numerical columns through the numerical pipeline.”

`("cat", categorical_pipeline, categorical_cols)`

means:

> “Send the categorical columns through the categorical pipeline.”

# STEP 5–6 — FIT and PREDICT

## 22. Connect preprocessing to Linear Regression

Now make one final workflow:

`raw X → preprocessing → Linear Regression`

In [ ]:
from sklearn.linear_model import LinearRegression

model = Pipeline([
    ("preprocess", preprocess),
    ("regressor", LinearRegression())
])

model

## 23. Fit the model

This one command triggers the whole training workflow.

In [ ]:
model.fit(X_train, y_train)

### What happens inside `fit()`

The workflow learns from the training data:

1. imputation medians
2. scaling parameters
3. one-hot categories
4. Linear Regression coefficients

You do not have to call each step manually.

## 24. Predict the test set

Again, give the model the raw `X_test`.

The Pipeline automatically applies the **training-learned** preprocessing before prediction.

In [ ]:
y_pred = model.predict(X_test)

pd.DataFrame({
    "Actual ($)": y_test.iloc[:5].values,
    "Predicted ($)": np.round(y_pred[:5])
})

### The key Pipeline idea

When you call:

`model.predict(X_test)`

you do **not** manually:

- impute test data
- scale test data
- one-hot encode test data

The Pipeline applies the stored training workflow automatically.

# Part E — Evaluate

## 25. RMSE — typical scale of prediction error

In [ ]:
from sklearn.metrics import mean_squared_error

rmse = mean_squared_error(
    y_test,
    y_pred
) ** 0.5

print(f"Test RMSE: ${rmse:,.0f}")

### Interpretation

RMSE is in the same unit as the target: **dollars**.

Lower is better.

Do not interpret RMSE as an individual prediction interval.
It summarizes error across the test set.

## 26. R² — variation accounted for relative to a mean baseline

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(
    y_test,
    y_pred
)

print(f"Test R²: {r2:.3f}")

### Interpretation

- R² = 1 → perfect predictions
- R² = 0 → no better, in squared-error terms, than predicting the mean target
- R² < 0 → worse than that baseline



## 27. Look at predictions, not only metrics

In [ ]:
plt.figure(figsize=(8, 8))

plt.scatter(
    y_test,
    y_pred,
    alpha=0.2
)

low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())

plt.plot([low, high], [low, high])

plt.xlabel("Actual median house value ($)", fontsize=16)
plt.ylabel("Predicted median house value ($)", fontsize=16)
plt.title("Actual vs predicted house values", fontsize=16)
plt.show()

### Interpretation

- points on the diagonal are perfect predictions
- distance from the line represents prediction error
- the plot shows patterns that one summary metric can hide

# Final map

| Stage | Main question | Decision / action |
|---|---|---|
| Problem | What are we predicting? | `median_house_value` |
| Inspect | What problems are in the raw data? | missing values + category + different scales |
| EDA | Which variables contain useful patterns? | inspect income, geography and target |
| Split | How do we protect final evaluation? | 80/20 stratified split |
| Prepare | What must be transformed? | impute, scale, one-hot encode |
| Pipeline | How do we repeat preparation safely? | keep steps in one workflow |
| ColumnTransformer | Which columns take which route? | numeric vs categorical paths |
| Fit | What is learned from training data? | preprocessing parameters + model |
| Predict | What happens to unseen X? | same stored preprocessing → prediction |
| Evaluate | How good are predictions? | RMSE + R² + actual-vs-predicted plot |

## Four statements to keep in mind

> **fit = learn**
> **transform = apply**
> **Pipeline = order of steps**
> **ColumnTransformer = routes for columns**

# One-minute workflow recall

Without looking back, complete this from memory:

1. **PROBLEM:** What are X and y?
2. **UNDERSTAND:** What did inspection/EDA reveal?
3. **SPLIT:** Why protect the test set? When was stratification useful here?
4. **PREPARE:** What did the numeric and categorical paths do?
5. **FIT:** What exactly was learned by `model.fit(X_train, y_train)`?
6. **PREDICT:** What happens internally when `model.predict(X_test)` is called?
7. **EVALUATE:** What do RMSE and R² tell us?

If you can explain these seven steps in your own words, you understand the workflow.